# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

GENRES = sorted(os.listdir(f"{DATA_ROOT}/genres_stems"))

STEMS = {
    "drums.wav": "drums",
    "vocals.wav": "vocals",
    "bass.wav": "bass",
    "other.wav": "other"
}

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']

GENRE_TO_TEST = 'rock'

SONG_INDEX = 110250 


In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    corrupted_count = 0
    small_count = 0
    large_count = 0

    for genre in GENRES:
        genre_path = os.path.join(root_dir, "genres_stems", genre)
        songs = sorted(os.listdir(genre_path))
        valid_songs = []

        for song in songs:
            song_path = os.path.join(genre_path, song)
            valid = True

            for stem_file in STEMS.keys():
                fp = os.path.join(song_path, stem_file)

                if not os.path.exists(fp):
                    valid = False
                else:
                    size = os.path.getsize(fp)

                    if size < 4 * 1024:
                        corrupted_count += 1
                        valid = False

                    size_mb = size / (1024 * 1024)
                    if size_mb < 5.0491:
                        small_count += 1
                    if size_mb > 5.0493:
                        large_count += 1

            if valid:
                valid_songs.append(song)

        rng.shuffle(valid_songs)
        split = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split]
        val_songs = valid_songs[split:]

        def add_to_dict(target_dict, song_list):
            for s in song_list:
                for stem_file in STEMS.keys():
                    stem_key = stem_file.replace(".wav", "")
                    target_dict[genre][stem_key].append(
                        os.path.join(genre_path, s, stem_file)
                    )

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    Q1 = corrupted_count + small_count
    Q2 = abs(large_count - small_count)
    Q3 = abs(len(train_dataset["reggae"]["drums"]) - len(val_dataset["country"]["vocals"]))

    return train_dataset, val_dataset,Q1,Q2,Q3


tr, val ,Q1,Q2,Q3= build_dataset(DATA_ROOT)


In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []

    total_files = sum(len(dataset_dict[g][s]) for g in dataset_dict for s in dataset_dict[g])

    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:

                y, _ = librosa.load(file_path, sr=sr)
                total_duration = len(y) / sr

                rms = librosa.feature.rms(
                    y=y,
                    frame_length=N_FFT,
                    hop_length=HOP_LENGTH
                )[0]

                rms_db = librosa.amplitude_to_db(rms, ref=np.max)
                silent_frames = rms_db < -top_db

                silence_lengths = []
                count = 0

                for val in silent_frames:
                    if val:
                        count += 1
                    else:
                        if count > 0:
                            silence_lengths.append(count * HOP_LENGTH / sr)
                            count = 0

                if count > 0:
                    silence_lengths.append(count * HOP_LENGTH / sr)

                if len(silence_lengths) == 0:
                    continue

                max_silence = max(silence_lengths)
                silence_type = []

                if silence_lengths[0] >= threshold_sec:
                    silence_type.append("start")

                if silence_lengths[-1] >= threshold_sec:
                    silence_type.append("end")

                if max_silence >= threshold_sec and not silence_type:
                    silence_type.append("middle")

                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(silence_type),
                        "File_Path": file_path
                    })

    df = pd.DataFrame(records)
    return df


df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

pivot_table = pd.pivot_table(df_silence, index="Genre", columns="Stem", aggfunc="size", fill_value=0)
pivot_table


In [ ]:
stems_audio = []
try:
    for key in STEM_KEYS:
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
        stems_audio.append(y)

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")


In [ ]:
stems_stack = np.vstack(stems_audio)

mix_raw = np.sum(stems_stack, axis=0)

rms_val = np.sqrt(np.mean(mix_raw ** 2))

max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."


In [ ]:
Q1=Q1
Q2=Q2
Q3=Q3

Q4 = len(df_silence)
Q5 = len(df_silence[df_silence["Stem"] == "vocals"])
Q6 = df_silence[df_silence["Stem"] == "vocals"]["Max_Silence_Sec"].mean()
Q7 = len(df_silence[(df_silence["Genre"] == "jazz") & (df_silence["Stem"] == "drums")])
Q8 = len(df_silence[(df_silence["Genre"] == "jazz") & 
                    (df_silence["Stem"] == "drums") & 
                    (df_silence["Silence_Location"] == "middle")])
Q9 = len(df_silence[(df_silence["Genre"] == "jazz") & 
                    (df_silence["Stem"] == "drums") & 
                    (df_silence["Max_Silence_Sec"] >= 10)])

Q10 = len(mix_raw)
Q11 = rms_val
Q12 = np.max(np.abs(mix_raw))

print("Q1:", Q1)
print("Q2:", Q2)
print("Q3:", Q3)
print("Q4:", Q4)
print("Q5:", Q5)
print("Q6:", Q6)
print("Q7:", Q7)
print("Q8:", Q8)
print("Q9:", Q9)
print("Q10:", Q10)
print("Q11:", Q11)
print("Q12:", Q12)
